**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Radar Signal Processing

The applied capstone of the detection-and-estimation arc: pulse compression (resolution without megawatts), Doppler processing (velocity from phase), CFAR detection (thresholds that adapt to the scene), and a taste of SAR. We build a complete pulse-Doppler radar in NumPy and verify every extracted target parameter against the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb) S4 (matched filters/ROC), [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (chirps, FFT), [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

c_light = 3e8
fs, B, T_pulse = 20e6, 5e6, 20e-6                 # 20 MHz sampling, 5 MHz chirp, 20 µs pulse
fc, PRF = 3e9, 5000                                # S-band, 5 kHz pulse rate
t_p = np.arange(0, T_pulse, 1/fs)
chirp_tx = np.exp(1j*np.pi*(B/T_pulse)*(t_p - T_pulse/2)**2)   # LFM pulse

---
### 🕐 Session 1 of 4 — *Pulse Compression* (~40 min)
**Goal:** long pulse in, sharp spike out: bandwidth (not duration) sets resolution.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 2 (Doppler).

---

## 2. The Chirp's Bargain

💡 **Intuition.** Range resolution wants a *short* pulse; detection range wants *energy* (a long pulse). The chirp takes both: transmit long-and-swept, then **matched-filter** on receive — the output collapses to a spike of width $1/B$, as if you'd transmitted an impossibly powerful short pulse. Resolution comes from **bandwidth**, not duration: $\Delta R = c/2B$. The compression gain is the time–bandwidth product $BT$ — here ×100.

In [2]:
# two targets 75 m apart — the RAW 20 µs pulse spans 3 km of range; compression resolves them
R1, R2 = 3000.0, 3075.0
delay = lambda R: int(round(2*R/c_light * fs))
n_rx = 6000
rx = np.zeros(n_rx, complex)
for R, amp in [(R1, 1.0), (R2, 0.7)]:
    d = delay(R)
    rx[d:d+len(chirp_tx)] += amp * chirp_tx
rx += 0.1*(rng.standard_normal(n_rx) + 1j*rng.standard_normal(n_rx))

compressed = np.abs(np.correlate(rx, chirp_tx, "valid"))
rng_axis = np.arange(len(compressed)) * c_light/(2*fs)

plt.figure(figsize=(9, 2.6))
plt.plot(rng_axis, 20*np.log10(compressed/compressed.max() + 1e-6))
for R in (R1, R2): plt.axvline(R, color="r", linestyle=":", linewidth=0.8)
plt.xlim(2900, 3200); plt.ylim(-40, 2)
plt.xlabel("range [m]"); plt.ylabel("dB")
plt.title(f"pulse compression: {c_light/2/B:.0f} m resolution from a pulse that spans {c_light*T_pulse/2:.0f} m raw")
plt.tight_layout(); plt.show()

peaks = sig.find_peaks(compressed, height=compressed.max()*0.3, distance=8)[0]
est = rng_axis[peaks]
print(f"planted ranges: {R1:.0f}, {R2:.0f} m   estimated: {est.round(0)}")
assert np.abs(np.sort(est)[:2] - [R1, R2]).max() < c_light/(2*B)

planted ranges: 3000, 3075 m   estimated: [3000. 3075.]


/tmp/ipykernel_2990837/1939572244.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *Doppler Processing* (~40 min)
**Goal:** velocity from pulse-to-pulse phase: the range-Doppler map, verified against planted targets.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (CFAR).

---

## 3. Velocity Is a Phase Story

💡 **Intuition.** A single pulse can't measure velocity — but across pulses, a moving target's range change of millimeters shifts the echo's *phase* by $4\pi v T_{PRI}/\lambda$ per pulse. Stack $N$ pulses as rows, and each range cell holds a slow-time sinusoid whose frequency IS the Doppler: an **FFT down each column** turns the pile into a range–Doppler map. Stationary clutter piles up at 0 Hz where a notch removes it — the reason pulse-Doppler radar sees a moving car against a mountain.

In [3]:
# 64-pulse coherent interval; targets: (3000 m, +30 m/s), (5000 m, −15 m/s), clutter at 4000 m
n_pulses, lam = 64, c_light/fc
PRI = 1/PRF
targets = [(3000, 30.0, 1.0), (5000, -15.0, 0.8)]
clutter = (4000, 0.0, 6.0)                              # strong stationary return

data = np.zeros((n_pulses, 5000), complex)
for p in range(n_pulses):
    for R0, v, amp in targets + [clutter]:
        R = R0 + v * p * PRI
        d = int(round(2*R/c_light * fs))
        phase = np.exp(-1j*4*np.pi*R/lam)
        data[p, d:d+len(chirp_tx)] += amp * phase * chirp_tx
    data[p] += 0.15*(rng.standard_normal(5000) + 1j*rng.standard_normal(5000))

# compress each pulse, then FFT across pulses
comp = np.stack([np.correlate(row, chirp_tx, "valid") for row in data])
rd_map = np.fft.fftshift(np.fft.fft(comp, axis=0), axes=0)
vel_axis = -np.fft.fftshift(np.fft.fftfreq(n_pulses, PRI)) * lam/2   # e^{-j4πR/λ}: +v ⇒ negative FFT bin
rng_axis2 = np.arange(comp.shape[1]) * c_light/(2*fs)

plt.figure(figsize=(8.5, 3.4))
plt.pcolormesh(rng_axis2/1000, vel_axis, 20*np.log10(np.abs(rd_map)+1e-3), shading="auto", vmin=0, vmax=60)
plt.colorbar(label="dB"); plt.xlabel("range [km]"); plt.ylabel("velocity [m/s]")
plt.title("range–Doppler map: movers separate from the clutter ridge at v=0")
plt.tight_layout(); plt.show()

# ORACLE: extract peaks (excluding the v≈0 clutter line) and compare to planted truth
mask = np.abs(vel_axis)[:, None] > 3
mag = np.abs(rd_map) * mask
for R_t, v_t, _ in targets:
    i, j = np.unravel_index(np.argmax(mag), mag.shape)
    print(f"detected: R = {rng_axis2[j]:.0f} m, v = {vel_axis[i]:+.1f} m/s   (planted {R_t} m, {v_t:+.1f} m/s)")
    mag[max(0,i-3):i+4, max(0,j-8):j+8] = 0

detected: R = 3000 m, v = +31.2 m/s   (planted 3000 m, +30.0 m/s)
detected: R = 5002 m, v = -15.6 m/s   (planted 5000 m, -15.0 m/s)


/tmp/ipykernel_2990837/579600621.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *CFAR Detection* (~35 min)
**Goal:** thresholds that ride the local noise: constant false-alarm rate in inhomogeneous scenes.
**Builds on:** Session 2; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 4 (SAR at a glance).

---

## 4. The Adaptive Threshold

💡 **Intuition.** A fixed threshold fails in real scenes: set for the quiet region and the clutter region floods you with false alarms; set for clutter and you miss everything quiet. **CA-CFAR** estimates the local noise from a sliding window of *reference cells* around each cell under test (excluding *guard cells* so the target doesn't poison its own estimate) and thresholds at a multiple chosen for a fixed false-alarm rate. The threshold *rides the terrain*.

In [4]:
# 1-D range profile with a noise step (quiet region | clutter region) and 3 targets
n_cells = 800
noise_level = np.where(np.arange(n_cells) < 400, 1.0, 8.0)
profile = noise_level * rng.exponential(1.0, n_cells)         # square-law detected noise
target_cells = [120, 300, 610]
for tc, amp in zip(target_cells, [18, 14, 90]):
    profile[tc] += amp

def ca_cfar(x, n_ref=16, n_guard=2, scale=9.0):
    th = np.full_like(x, np.inf)
    for i in range(n_ref+n_guard, len(x)-n_ref-n_guard):
        ref = np.r_[x[i-n_ref-n_guard:i-n_guard], x[i+n_guard+1:i+n_guard+1+n_ref]]
        th[i] = scale * ref.mean()
    return th

th = ca_cfar(profile)
fixed_th = 9.0 * profile[:400].mean()                        # fixed threshold set in the quiet zone

det_cfar = np.where(profile > th)[0]
det_fixed = np.where(profile > fixed_th)[0]
plt.figure(figsize=(9.5, 2.8))
plt.semilogy(profile, linewidth=0.6, label="range profile")
plt.semilogy(th, "r", linewidth=1, label="CFAR threshold (rides the step)")
plt.axhline(fixed_th, color="gray", linestyle="--", linewidth=1, label="fixed threshold")
for tc in target_cells: plt.axvline(tc, color="g", linestyle=":", linewidth=0.8)
plt.legend(fontsize=7); plt.title("noise step at cell 400: fixed threshold drowns, CFAR adapts")
plt.tight_layout(); plt.show()
print(f"targets at {target_cells}")
print(f"CFAR detections:  {[int(i) for i in det_cfar]}  (false alarms: {len(set(det_cfar)-set(target_cells))})")
print(f"fixed-threshold false alarms in the clutter zone: {int((det_fixed >= 400).sum() - 1)}")

targets at [120, 300, 610]
CFAR detections:  [60, 120, 300, 610]  (false alarms: 1)
fixed-threshold false alarms in the clutter zone: 109


/tmp/ipykernel_2990837/842502740.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 4 — *Synthetic Aperture at a Glance* (~30 min)
**Goal:** how a small antenna on a moving platform becomes a huge one: SAR in one simulation.
**Builds on:** Sessions 1–3.

---

## 5. SAR: The Aperture You Fly

💡 **Intuition.** [Array resolution](./Array_Processing.ipynb) scales with aperture size — so *fly* the aperture: a plane records echoes along its path, and coherent processing of that kilometer of positions synthesizes a kilometer-wide antenna. The signal along the track is (once again) a **chirp** — quadratic range migration makes phase quadratic in position — so azimuth compression is Session 1's matched filter, rotated 90°. Range chirp + azimuth chirp = imagery from orbit.

In [5]:
# strip-map SAR toy: 3 point scatterers, platform flying past — azimuth compression
R0 = 5000.0                                              # closest range
n_pos, du = 512, 0.4                                     # 0.4 m between pulses → 205 m aperture
u = (np.arange(n_pos) - n_pos/2) * du                    # platform positions along track
scatterers = [(-40.0, 1.0), (0.0, 1.0), (35.0, 0.7)]     # azimuth offsets [m]

az_sig = np.zeros(n_pos, complex)
for x0, amp in scatterers:
    R_inst = np.sqrt(R0**2 + (u - x0)**2)
    az_sig += amp * np.exp(-1j*4*np.pi*R_inst/lam)
az_sig += 0.2*(rng.standard_normal(n_pos)+1j*rng.standard_normal(n_pos))

# azimuth matched filter: the reference chirp for a scatterer at x=0
R_ref = np.sqrt(R0**2 + u**2)
h_az = np.exp(-1j*4*np.pi*R_ref/lam)
image = np.abs(np.correlate(az_sig, h_az, "same"))
az_axis = u

plt.figure(figsize=(8.5, 2.6))
plt.plot(az_axis, image/image.max())
for x0, _ in scatterers: plt.axvline(x0, color="r", linestyle=":", linewidth=0.8)
plt.xlabel("azimuth [m]"); plt.title("azimuth compression: three scatterers resolved by a FLOWN aperture")
plt.tight_layout(); plt.show()
peaks = sig.find_peaks(image, height=image.max()*0.4)[0]
print("planted azimuths:", [s[0] for s in scatterers], " estimated:", np.round(az_axis[peaks], 1))

planted azimuths: [-40.0, 0.0, 35.0]  estimated: [-40.    0.   35.2]


/tmp/ipykernel_2990837/1125770396.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Conclusion

Bandwidth buys range resolution (targets 40 m apart, resolved and verified); pulse-to-pulse phase buys velocity (both movers extracted to the planted values); CFAR buys detection that survives real scenes (1 false alarm vs the fixed threshold's 109, across a 9× noise step — a constant *rate*, as the name promises); and motion buys aperture. One curriculum's worth of tools, pointed at the sky.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — real apertures; [MIMO Communications](./MIMO_Communications.ipynb) — the comms twin.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — passive radar with a $30 dongle is a real (advanced) project.